# Amazon ML Challenge 2026 — FINE TUNING

Clean submission-oriented V1 pipeline.

**Frozen:** blocker definitions, 5 features, Logistic Regression, threshold `0.95`.

**Engineering:** reusable Parquet artifacts, DuckDB for large candidate operations, bounded smoke test before full inference.

In [10]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd
import polars as pl

from pathlib import Path

from rapidfuzz.fuzz import ratio

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    precision_score,
    recall_score,
    fbeta_score
)

print("Environment ready.")

Environment ready.


In [11]:
# ============================================================
# 2. LOCATE TUNING BUNDLE
# ============================================================

INPUT_DIR = Path("/kaggle/input")

print("Files available in Kaggle input:\n")

for path in INPUT_DIR.rglob("*"):
    if path.is_file():
        print(path)

Files available in Kaggle input:

/kaggle/input/datasets/abhilashpalit/ml2026/test_source2.tsv
/kaggle/input/datasets/abhilashpalit/ml2026/test_source3.tsv
/kaggle/input/datasets/abhilashpalit/ml2026/train_ground_truth.tsv
/kaggle/input/datasets/abhilashpalit/ml2026/train_source3.tsv
/kaggle/input/datasets/abhilashpalit/ml2026/test_source1.tsv
/kaggle/input/datasets/abhilashpalit/ml2026/train_source2.tsv
/kaggle/input/datasets/abhilashpalit/ml2026/train_source1.tsv
/kaggle/input/datasets/abhilashpalit/picklebundle/amazon_ml_tuning_bundle.pkl
/kaggle/input/datasets/abhilashpalit/amazonml2026v01/v1_model.joblib


In [17]:
# ============================================================
# 3. LOAD TUNING BUNDLE
# ============================================================

BUNDLE_PATH = Path(
    "/kaggle/input/datasets/abhilashpalit/picklebundle/amazon_ml_tuning_bundle.pkl"
)

if not BUNDLE_PATH.exists():
    raise FileNotFoundError(
        f"Tuning bundle not found:\n{BUNDLE_PATH}"
    )

with open(BUNDLE_PATH, "rb") as f:
    bundle = pickle.load(f)

print("Tuning bundle loaded successfully.")
print("Size:", f"{BUNDLE_PATH.stat().st_size / 1024**2:.2f} MB")

Tuning bundle loaded successfully.
Size: 77.78 MB


In [18]:
# ============================================================
# 4. EXTRACT EXPORTED OBJECTS
# ============================================================

training_enriched = bundle["training_enriched"]
training_features = bundle["training_features"]

X_train_v1 = bundle["X_train"]
X_val_v1 = bundle["X_val"]

y_train = bundle["y_train"]
y_val = bundle["y_val"]

train_idx = bundle["train_idx"]
val_idx = bundle["val_idx"]

V1_FEATURES = bundle["v1_features"]
V1_THRESHOLD = bundle["v1_threshold"]

print("============================================")
print("TUNING DATA LOADED")
print("============================================")

print("training_enriched :", training_enriched.shape)
print("training_features :", training_features.shape)

print("X_train           :", X_train_v1.shape)
print("X_val             :", X_val_v1.shape)

print("y_train           :", y_train.shape)
print("y_val             :", y_val.shape)

print()
print("V1 features:")
print(V1_FEATURES)

print()
print("V1 threshold:", V1_THRESHOLD)

TUNING DATA LOADED
training_enriched : (172458, 9)
training_features : (172458, 16)
X_train           : (137966, 5)
X_val             : (34492, 5)
y_train           : (137966,)
y_val             : (34492,)

V1 features:
['name_edit_similarity', 'name_token_jaccard', 'address_edit_similarity', 'address_token_jaccard', 'country_match']

V1 threshold: 0.95


In [19]:
# ============================================================
# 5. VERIFY FIXED TRAIN / VALIDATION SPLIT
# ============================================================

print("Total training rows :", len(train_idx) + len(val_idx))
print("Train rows          :", len(train_idx))
print("Validation rows     :", len(val_idx))

print()
print("Unique train indices :", len(np.unique(train_idx)))
print("Unique val indices   :", len(np.unique(val_idx)))

overlap = np.intersect1d(train_idx, val_idx)

print("Train/val overlap    :", len(overlap))

if len(overlap) != 0:
    raise RuntimeError("Train and validation indices overlap!")

if len(train_idx) + len(val_idx) != training_enriched.height:
    raise RuntimeError(
        "Train + validation indices do not cover training_enriched."
    )

print("\n✓ Fixed split verified.")

Total training rows : 172458
Train rows          : 137966
Validation rows     : 34492

Unique train indices : 137966
Unique val indices   : 34492
Train/val overlap    : 0

✓ Fixed split verified.


In [20]:
# ============================================================
# 6_A0. LOAD FROZEN V1 MODEL
# ============================================================

MODEL_PATH = Path(
    "/kaggle/input/datasets/abhilashpalit/amazonml2026v01/v1_model.joblib"
)

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"V1 model not found:\n{MODEL_PATH}"
    )

v1_model_frozen = joblib.load(MODEL_PATH)

print("Frozen V1 model loaded successfully.")
print()
print(v1_model_frozen)

Frozen V1 model loaded successfully.

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])


In [21]:
# ============================================================
# 6. TRAIN V1 LOGISTIC REGRESSION
# ============================================================

v1_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

v1_model.fit(
    X_train_v1,
    y_train
)

print("V1 model trained.")

V1 model trained.


In [22]:
# ============================================================
# 7. V1 VALIDATION PROBABILITIES
# ============================================================

v1_val_prob = v1_model.predict_proba(
    X_val_v1
)[:, 1]

print("Validation predictions generated.")

print("Count :", len(v1_val_prob))
print("Min   :", v1_val_prob.min())
print("Max   :", v1_val_prob.max())
print("Mean  :", v1_val_prob.mean())

Validation predictions generated.
Count : 34492
Min   : 3.5104234475412015e-05
Max   : 0.9999999758547297
Mean  : 0.26657050381633657


In [23]:
# ============================================================
# 8. V1 THRESHOLD SEARCH
# ============================================================

thresholds = np.arange(
    0.50,
    1.001,
    0.01
)

v1_results = []

for threshold in thresholds:

    pred = (
        v1_val_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        pred,
        zero_division=0
    )

    f05 = fbeta_score(
        y_val,
        pred,
        beta=0.5,
        zero_division=0
    )

    v1_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f05": f05
    })

v1_results = pd.DataFrame(v1_results)

best_v1 = v1_results.loc[
    v1_results["f05"].idxmax()
]

print("============================================")
print("V1 THRESHOLD SEARCH")
print("============================================")

print(best_v1)

V1 THRESHOLD SEARCH
threshold    0.950000
precision    0.959824
recall       0.605185
f05          0.859134
Name: 45, dtype: float64


In [33]:
# ============================================================
# FROZEN V1 vs FRESHLY TRAINED V1
# ============================================================

frozen_prob = v1_model_frozen.predict_proba(
    X_val_v1
)[:, 1]

fresh_prob = v1_model.predict_proba(
    X_val_v1
)[:, 1]

print("Frozen probability statistics:")
print("  min :", frozen_prob.min())
print("  max :", frozen_prob.max())
print("  mean:", frozen_prob.mean())

print("\nFresh probability statistics:")
print("  min :", fresh_prob.min())
print("  max :", fresh_prob.max())
print("  mean:", fresh_prob.mean())

print("\nProbability difference:")
print(
    "  mean absolute difference:",
    np.mean(np.abs(frozen_prob - fresh_prob))
)

print(
    "  max absolute difference:",
    np.max(np.abs(frozen_prob - fresh_prob))
)

Frozen probability statistics:
  min : 3.228159775830668e-05
  max : 0.999999974013202
  mean: 0.26609542389308494

Fresh probability statistics:
  min : 3.5104234475412015e-05
  max : 0.9999999758547297
  mean: 0.26657050381633657

Probability difference:
  mean absolute difference: 0.0013679212575785982
  max absolute difference: 0.023348861682586763


In [35]:
# ============================================================
# COMPARE PREDICTIONS AT FROZEN THRESHOLD
# ============================================================

frozen_pred = (
    frozen_prob >= 0.95
).astype(int)

fresh_pred = (
    fresh_prob >= 0.95
).astype(int)

print("Frozen positive predictions:",
      frozen_pred.sum())

print("Fresh positive predictions :",
      fresh_pred.sum())

print(
    "Prediction disagreements:",
    np.sum(frozen_pred != fresh_pred)
)

Frozen positive predictions: 3143
Fresh positive predictions : 3186
Prediction disagreements: 49


In [36]:
# ============================================================
# FROZEN V1 METRICS AT EXACT THRESHOLD = 0.95
# ============================================================

frozen_pred_095 = (
    frozen_prob >= 0.95
).astype(int)

frozen_precision = precision_score(
    y_val,
    frozen_pred_095,
    zero_division=0
)

frozen_recall = recall_score(
    y_val,
    frozen_pred_095,
    zero_division=0
)

frozen_f05 = fbeta_score(
    y_val,
    frozen_pred_095,
    beta=0.5,
    zero_division=0
)

print("=" * 50)
print("FROZEN V1 @ THRESHOLD 0.95")
print("=" * 50)

print(f"Precision : {frozen_precision:.6%}")
print(f"Recall    : {frozen_recall:.6%}")
print(f"F0.5      : {frozen_f05:.6%}")

print()
print("Positive predictions:", frozen_pred_095.sum())
print("Actual positives    :", np.sum(np.asarray(y_val) == 1))

FROZEN V1 @ THRESHOLD 0.95
Precision : 96.913777%
Recall    : 60.281021%
F0.5      : 86.411348%

Positive predictions: 3143
Actual positives    : 5053


In [24]:
# ============================================================
# 9. V1 CHECKPOINT
# ============================================================

v1_threshold = float(best_v1["threshold"])

v1_pred = (
    v1_val_prob >= v1_threshold
).astype(int)

v1_precision = precision_score(
    y_val,
    v1_pred,
    zero_division=0
)

v1_recall = recall_score(
    y_val,
    v1_pred,
    zero_division=0
)

v1_f05 = fbeta_score(
    y_val,
    v1_pred,
    beta=0.5,
    zero_division=0
)

print("=" * 50)
print("FROZEN V1 REFERENCE")
print("=" * 50)

print(f"Threshold : {v1_threshold:.2f}")
print(f"Precision : {v1_precision:.4%}")
print(f"Recall    : {v1_recall:.4%}")
print(f"F0.5      : {v1_f05:.4%}")

FROZEN V1 REFERENCE
Threshold : 0.95
Precision : 95.9824%
Recall    : 60.5185%
F0.5      : 85.9134%


In [25]:
# ============================================================
# 10. V2 FEATURE ENGINEERING
# ============================================================

import re


def normalize_text(value):
    if value is None:
        return ""

    value = str(value).lower()

    value = re.sub(
        r"[^a-z0-9\s]",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def normalize_legal_name(value):
    value = normalize_text(value)

    legal_suffixes = [
        "private limited",
        "pvt ltd",
        "pvt limited",
        "private ltd",
        "limited",
        "ltd",
        "llc",
        "incorporated",
        "inc",
        "corp",
        "corporation",
        "company",
        "co"
    ]

    tokens = value.split()

    # Remove suffixes only from the end
    while tokens:

        current = " ".join(tokens)

        removed = False

        for suffix in legal_suffixes:

            suffix_tokens = suffix.split()

            if (
                len(tokens) >= len(suffix_tokens)
                and tokens[-len(suffix_tokens):]
                == suffix_tokens
            ):
                tokens = tokens[:-len(suffix_tokens)]
                removed = True
                break

        if not removed:
            break

    return " ".join(tokens)


def safe_length_ratio(a, b):

    a = a or ""
    b = b or ""

    if not a or not b:
        return 0.0

    shorter = min(len(a), len(b))
    longer = max(len(a), len(b))

    if longer == 0:
        return 0.0

    return shorter / longer


def exact_normalized(a, b):

    a = normalize_text(a)
    b = normalize_text(b)

    if not a or not b:
        return 0

    return int(a == b)


def exact_legal_name(a, b):

    a = normalize_legal_name(a)
    b = normalize_legal_name(b)

    if not a or not b:
        return 0

    return int(a == b)


print("V2 feature functions ready.")

V2 feature functions ready.


In [26]:
# ============================================================
# 11. BUILD V2 FEATURE TABLE
# ============================================================

v2 = (
    training_enriched
    .with_columns([

        # ====================================================
        # NAME
        # ====================================================

        pl.struct([
            "s1_name",
            "match_name"
        ])
        .map_elements(
            lambda x: exact_normalized(
                x["s1_name"],
                x["match_name"]
            ),
            return_dtype=pl.Int8
        )
        .alias("name_exact_normalized"),

        pl.struct([
            "s1_name",
            "match_name"
        ])
        .map_elements(
            lambda x: exact_legal_name(
                x["s1_name"],
                x["match_name"]
            ),
            return_dtype=pl.Int8
        )
        .alias("name_exact_legal"),

        pl.struct([
            "s1_name",
            "match_name"
        ])
        .map_elements(
            lambda x: safe_length_ratio(
                x["s1_name"],
                x["match_name"]
            ),
            return_dtype=pl.Float64
        )
        .alias("name_length_ratio"),

        # ====================================================
        # ADDRESS
        # ====================================================

        pl.struct([
            "s1_address",
            "match_address"
        ])
        .map_elements(
            lambda x: exact_normalized(
                x["s1_address"],
                x["match_address"]
            ),
            return_dtype=pl.Int8
        )
        .alias("address_exact_normalized"),

        pl.struct([
            "s1_address",
            "match_address"
        ])
        .map_elements(
            lambda x: safe_length_ratio(
                x["s1_address"],
                x["match_address"]
            ),
            return_dtype=pl.Float64
        )
        .alias("address_length_ratio"),

        # ====================================================
        # MISSINGNESS
        # ====================================================

        (
            pl.col("s1_name")
            .fill_null("")
            .str.strip_chars()
            == ""
        )
        .cast(pl.Int8)
        .alias("s1_name_missing"),

        (
            pl.col("match_name")
            .fill_null("")
            .str.strip_chars()
            == ""
        )
        .cast(pl.Int8)
        .alias("match_name_missing"),

        (
            pl.col("s1_address")
            .fill_null("")
            .str.strip_chars()
            == ""
        )
        .cast(pl.Int8)
        .alias("s1_address_missing"),

        (
            pl.col("match_address")
            .fill_null("")
            .str.strip_chars()
            == ""
        )
        .cast(pl.Int8)
        .alias("match_address_missing"),
    ])
)

print("V2 features created.")
print("Rows:", v2.height)
print("Columns:", len(v2.columns))

print("\nNew V2 columns:")
print([
    c for c in v2.columns
    if c not in training_enriched.columns
])

V2 features created.
Rows: 172458
Columns: 18

New V2 columns:
['name_exact_normalized', 'name_exact_legal', 'name_length_ratio', 'address_exact_normalized', 'address_length_ratio', 's1_name_missing', 'match_name_missing', 's1_address_missing', 'match_address_missing']


In [27]:
# ============================================================
# 12. COMBINE V1 + V2 FEATURES
# ============================================================

V1_FEATURES = [
    "name_edit_similarity",
    "name_token_jaccard",
    "address_edit_similarity",
    "address_token_jaccard",
    "country_match"
]

V2_NEW_FEATURES = [
    "name_exact_normalized",
    "name_exact_legal",
    "name_length_ratio",
    "address_exact_normalized",
    "address_length_ratio",
    "s1_name_missing",
    "match_name_missing",
    "s1_address_missing",
    "match_address_missing"
]

v2_feature_table = v2.with_columns([
    training_features[col]
    for col in V1_FEATURES
])

V2_FEATURES = (
    V1_FEATURES
    + V2_NEW_FEATURES
)

print("Total V2 features:", len(V2_FEATURES))

print("\nV2 features:")
for feature in V2_FEATURES:
    print(" ", feature)

Total V2 features: 14

V2 features:
  name_edit_similarity
  name_token_jaccard
  address_edit_similarity
  address_token_jaccard
  country_match
  name_exact_normalized
  name_exact_legal
  name_length_ratio
  address_exact_normalized
  address_length_ratio
  s1_name_missing
  match_name_missing
  s1_address_missing
  match_address_missing


In [28]:
# ============================================================
# 13. VERIFY V2 FEATURES
# ============================================================

print("Rows:", v2_feature_table.height)

missing_features = [
    feature
    for feature in V2_FEATURES
    if feature not in v2_feature_table.columns
]

print("Missing features:", missing_features)

if missing_features:
    raise RuntimeError(
        f"Missing V2 features: {missing_features}"
    )

print("\nV2 feature sample:")
print(
    v2_feature_table
    .select([
        "label",
        *V2_FEATURES
    ])
    .head(10)
)

print("\n✓ V2 feature table verified.")

Rows: 172458
Missing features: []

V2 feature sample:
shape: (10, 15)
┌───────┬────────────┬────────────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ label ┆ name_edit_ ┆ name_token ┆ address_ed ┆ … ┆ s1_name_m ┆ match_nam ┆ s1_addres ┆ match_add │
│ ---   ┆ similarity ┆ _jaccard   ┆ it_similar ┆   ┆ issing    ┆ e_missing ┆ s_missing ┆ ress_miss │
│ i32   ┆ ---        ┆ ---        ┆ ity        ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ing       │
│       ┆ f64        ┆ f64        ┆ ---        ┆   ┆ i8        ┆ i8        ┆ i8        ┆ ---       │
│       ┆            ┆            ┆ f64        ┆   ┆           ┆           ┆           ┆ i8        │
╞═══════╪════════════╪════════════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1     ┆ 0.533333   ┆ 0.2        ┆ 0.64       ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0         │
│ 1     ┆ 0.113208   ┆ 0.0        ┆ 0.897059   ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0         │
│ 1     ┆ 0.596491   

In [30]:
# ============================================================
# V2 FEATURE DISTRIBUTIONS
# ============================================================

print("V2 FEATURE DISTRIBUTIONS")
print("=" * 60)

for feature in V2_FEATURES:
    series = v2_feature_table[feature]

    print(f"\n{feature}")
    print(f"  dtype       : {series.dtype}")
    print(f"  unique      : {series.n_unique()}")
    print(f"  zero count  : {(series == 0).sum()}")
    print(f"  zero %      : {(series == 0).mean():.2%}")
    
    if series.dtype in [
        pl.Int8,
        pl.Int16,
        pl.Int32,
        pl.Int64,
        pl.Float32,
        pl.Float64
    ]:
        print(f"  mean        : {series.mean()}")
        print(f"  min         : {series.min()}")
        print(f"  max         : {series.max()}")

V2 FEATURE DISTRIBUTIONS

name_edit_similarity
  dtype       : Float64
  unique      : 1277
  zero count  : 440
  zero %      : 0.26%
  mean        : 0.3479411475614359
  min         : 0.0
  max         : 1.0

name_token_jaccard
  dtype       : Float64
  unique      : 38
  zero count  : 133067
  zero %      : 77.16%
  mean        : 0.09772798651917045
  min         : 0.0
  max         : 1.0

address_edit_similarity
  dtype       : Float64
  unique      : 6125
  zero count  : 632
  zero %      : 0.37%
  mean        : 0.3673917025445498
  min         : 0.0
  max         : 1.0

address_token_jaccard
  dtype       : Float64
  unique      : 252
  zero count  : 67785
  zero %      : 39.31%
  mean        : 0.09040204758856747
  min         : 0.0
  max         : 1.0

country_match
  dtype       : Int8
  unique      : 2
  zero count  : 15252
  zero %      : 8.84%
  mean        : 0.9115610757401802
  min         : 0
  max         : 1

name_exact_normalized
  dtype       : Int8
  unique      : 2


In [32]:
# ============================================================
# EXACT MATCH FEATURE CHECK
# ============================================================

exact_features = [
    "name_exact_normalized",
    "name_exact_legal",
    "address_exact_normalized"
]

for feature in exact_features:
    
    counts = (
        v2_feature_table
        .group_by(feature)
        .len()
        .sort(feature)
    )
    
    print(f"\n{feature}")
    print(counts)


name_exact_normalized
shape: (2, 2)
┌───────────────────────┬────────┐
│ name_exact_normalized ┆ len    │
│ ---                   ┆ ---    │
│ i8                    ┆ u32    │
╞═══════════════════════╪════════╡
│ 0                     ┆ 165079 │
│ 1                     ┆ 7379   │
└───────────────────────┴────────┘

name_exact_legal
shape: (2, 2)
┌──────────────────┬────────┐
│ name_exact_legal ┆ len    │
│ ---              ┆ ---    │
│ i8               ┆ u32    │
╞══════════════════╪════════╡
│ 0                ┆ 158613 │
│ 1                ┆ 13845  │
└──────────────────┴────────┘

address_exact_normalized
shape: (2, 2)
┌──────────────────────────┬────────┐
│ address_exact_normalized ┆ len    │
│ ---                      ┆ ---    │
│ i8                       ┆ u32    │
╞══════════════════════════╪════════╡
│ 0                        ┆ 170081 │
│ 1                        ┆ 2377   │
└──────────────────────────┴────────┘


In [29]:
# ============================================================
# 14. BUILD V2 TRAIN / VALIDATION SETS
# ============================================================

X_v2 = (
    v2_feature_table
    .select(V2_FEATURES)
    .to_pandas()
)

y_v2 = (
    v2_feature_table["label"]
    .to_pandas()
)

X_train_v2 = X_v2.iloc[train_idx].copy()
X_val_v2 = X_v2.iloc[val_idx].copy()

y_train_v2 = y_v2.iloc[train_idx].copy()
y_val_v2 = y_v2.iloc[val_idx].copy()

print("V2 TRAIN:")
print("X:", X_train_v2.shape)
print("y:", y_train_v2.shape)

print("\nV2 VALIDATION:")
print("X:", X_val_v2.shape)
print("y:", y_val_v2.shape)

V2 TRAIN:
X: (137966, 14)
y: (137966,)

V2 VALIDATION:
X: (34492, 14)
y: (34492,)


In [37]:
# ============================================================
# 15. TRAIN V2 LOGISTIC REGRESSION
# ============================================================

v2_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

v2_model.fit(
    X_train_v2,
    y_train_v2
)

print("V2 Logistic Regression trained.")

V2 Logistic Regression trained.


In [38]:
# ============================================================
# 16. V2 VALIDATION PROBABILITIES
# ============================================================

v2_val_prob = v2_model.predict_proba(
    X_val_v2
)[:, 1]

print("V2 predictions generated.")

print("Count :", len(v2_val_prob))
print("Min   :", v2_val_prob.min())
print("Max   :", v2_val_prob.max())
print("Mean  :", v2_val_prob.mean())

V2 predictions generated.
Count : 34492
Min   : 1.177591132684059e-06
Max   : 0.9999999999999138
Mean  : 0.2407994156977465


In [39]:
# ============================================================
# 17. V2 THRESHOLD SEARCH
# ============================================================

v2_results = []

for threshold in thresholds:

    pred = (
        v2_val_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_val_v2,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_val_v2,
        pred,
        zero_division=0
    )

    f05 = fbeta_score(
        y_val_v2,
        pred,
        beta=0.5,
        zero_division=0
    )

    v2_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f05": f05
    })

v2_results = pd.DataFrame(v2_results)

best_v2 = v2_results.loc[
    v2_results["f05"].idxmax()
]

print("============================================")
print("V2 THRESHOLD SEARCH")
print("============================================")

print(best_v2)

V2 THRESHOLD SEARCH
threshold    0.920000
precision    0.919753
recall       0.707698
f05          0.867751
Name: 42, dtype: float64


In [40]:
# ============================================================
# 18. V1 vs V2
# ============================================================

comparison = pd.DataFrame([
    {
        "experiment": "V1",
        "features": len(V1_FEATURES),
        "threshold": best_v1["threshold"],
        "precision": best_v1["precision"],
        "recall": best_v1["recall"],
        "f05": best_v1["f05"],
    },
    {
        "experiment": "V2",
        "features": len(V2_FEATURES),
        "threshold": best_v2["threshold"],
        "precision": best_v2["precision"],
        "recall": best_v2["recall"],
        "f05": best_v2["f05"],
    }
])

print(comparison.to_string(index=False))

experiment  features  threshold  precision   recall      f05
        V1         5       0.95   0.959824 0.605185 0.859134
        V2        14       0.92   0.919753 0.707698 0.867751


In [41]:
# ============================================================
# 19. SAVE V2 EXPERIMENT RESULTS
# ============================================================

experiment_dir = Path("/kaggle/working/fine_tuning")
experiment_dir.mkdir(
    parents=True,
    exist_ok=True
)

comparison.to_csv(
    experiment_dir / "v1_v2_comparison.csv",
    index=False
)

v1_results.to_csv(
    experiment_dir / "v1_threshold_results.csv",
    index=False
)

v2_results.to_csv(
    experiment_dir / "v2_threshold_results.csv",
    index=False
)

print("Experiment results saved to:")
print(experiment_dir)

Experiment results saved to:
/kaggle/working/fine_tuning


In [42]:
# ============================================================
# 17. RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

rf_model.fit(
    X_train_v2,
    y_train_v2
)

rf_prob = rf_model.predict_proba(
    X_val_v2
)[:, 1]

print("Random Forest trained.")

Random Forest trained.


In [44]:
# ============================================================
# THRESHOLD EVALUATION FUNCTION
# ============================================================

def evaluate_thresholds(y_true, probabilities, name="MODEL"):

    results = []

    for threshold in np.arange(0.50, 1.001, 0.01):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_true,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_true,
            predictions,
            zero_division=0
        )

        f05 = fbeta_score(
            y_true,
            predictions,
            beta=0.5,
            zero_division=0
        )

        results.append({
            "model": name,
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "f05": f05
        })

    results = pd.DataFrame(results)

    best = results.loc[
        results["f05"].idxmax()
    ]

    return results, best


print("Threshold evaluation function ready.")

Threshold evaluation function ready.


In [45]:
# ============================================================
# 18. RANDOM FOREST THRESHOLD SEARCH
# ============================================================

rf_results, rf_best = evaluate_thresholds(
    y_val_v2,
    rf_prob,
    "V2 Random Forest"
)

print(rf_best)

model        V2 Random Forest
threshold                0.69
precision            0.972422
recall               0.858302
f05                  0.947233
Name: 19, dtype: object


In [46]:
# ============================================================
# 19. XGBOOST
# ============================================================

import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

xgb_model.fit(
    X_train_v2,
    y_train_v2
)

xgb_prob = xgb_model.predict_proba(
    X_val_v2
)[:, 1]

print("XGBoost trained.")

XGBoost trained.


In [47]:
# ============================================================
# 20. XGBOOST THRESHOLD SEARCH
# ============================================================

xgb_results, xgb_best = evaluate_thresholds(
    y_val_v2,
    xgb_prob,
    "V2 XGBoost"
)

print(xgb_best)

model        V2 XGBoost
threshold          0.81
precision      0.980547
recall         0.837918
f05            0.948264
Name: 31, dtype: object


In [49]:
# ============================================================
# 19. XGBOOST - 2
# ============================================================
xgb_model_2 = xgb.XGBClassifier(
    n_estimators=700,
    max_depth=5,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

xgb_model_2.fit(
    X_train_v2,
    y_train_v2
)

xgb_prob_2 = xgb_model_2.predict_proba(
    X_val_v2
)[:, 1]

xgb_results_2, xgb_best_2 = evaluate_thresholds(
    y_val_v2,
    xgb_prob_2,
    "V2 XGBoost 2"
)

print(xgb_best_2)

model        V2 XGBoost 2
threshold            0.69
precision        0.974111
recall           0.856323
f05               0.94803
Name: 19, dtype: object


In [50]:
# ============================================================
# 19. XGBOOST - 3
# ============================================================
xgb_model_3 = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

xgb_model_3.fit(
    X_train_v2,
    y_train_v2
)

xgb_prob_3 = xgb_model_3.predict_proba(
    X_val_v2
)[:, 1]

xgb_results_3, xgb_best_3 = evaluate_thresholds(
    y_val_v2,
    xgb_prob_3,
    "V2 XGBoost 3"
)

print(xgb_best_3)

model        V2 XGBoost 3
threshold            0.75
precision        0.976174
recall           0.851375
f05              0.948371
Name: 25, dtype: object


In [51]:
# ============================================================
# FINAL XGBOOST MODEL
# ============================================================

FINAL_THRESHOLD = 0.75

final_xgb = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

print("Training final XGBoost on ALL labeled pairs...")

final_xgb.fit(
    X_v2,
    y_v2
)

print("Final XGBoost trained.")
print()
print("Features :", len(V2_FEATURES))
print("Rows     :", len(X_v2))
print("Threshold:", FINAL_THRESHOLD)

Training final XGBoost on ALL labeled pairs...
Final XGBoost trained.

Features : 14
Rows     : 172458
Threshold: 0.75


In [52]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================

FINAL_MODEL_PATH = Path(
    "/kaggle/working/final_xgb_v2.joblib"
)

joblib.dump(
    final_xgb,
    FINAL_MODEL_PATH
)

print("Saved:")
print(FINAL_MODEL_PATH)

print(
    f"Size: {FINAL_MODEL_PATH.stat().st_size / 1024**2:.2f} MB"
)

Saved:
/kaggle/working/final_xgb_v2.joblib
Size: 2.80 MB


In [53]:
# ============================================================
# EXPERIMENT 1: FINE THRESHOLD SEARCH
# ============================================================

fine_results = []

for threshold in np.arange(0.65, 0.851, 0.0025):

    predictions = (
        xgb_prob_3 >= threshold
    ).astype(int)

    precision = precision_score(
        y_val_v2,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_val_v2,
        predictions,
        zero_division=0
    )

    f05 = fbeta_score(
        y_val_v2,
        predictions,
        beta=0.5,
        zero_division=0
    )

    fine_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f05": f05
    })

fine_results = pd.DataFrame(fine_results)

best_fine = fine_results.loc[
    fine_results["f05"].idxmax()
]

print(best_fine)

threshold    0.795000
precision    0.978246
recall       0.845438
f05          0.948448
Name: 58, dtype: float64


In [54]:
# ============================================================
# EXPERIMENT 2: XGBOOST DEPTH 8 / CHILD WEIGHT 1
# ============================================================

xgb_model_4 = xgb.XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=1,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

xgb_model_4.fit(
    X_train_v2,
    y_train_v2
)

xgb_prob_4 = xgb_model_4.predict_proba(
    X_val_v2
)[:, 1]

xgb_results_4, xgb_best_4 = evaluate_thresholds(
    y_val_v2,
    xgb_prob_4,
    "V2 XGBoost 4"
)

print(xgb_best_4)

model        V2 XGBoost 4
threshold            0.72
precision        0.975192
recall           0.855729
f05              0.948703
Name: 22, dtype: object


In [55]:
# ============================================================
# EXPERIMENT 3: REGULARIZED XGBOOST
# ============================================================

xgb_model_5 = xgb.XGBClassifier(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0.05,
    reg_lambda=2.0,
    reg_alpha=0.05,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

xgb_model_5.fit(
    X_train_v2,
    y_train_v2
)

xgb_prob_5 = xgb_model_5.predict_proba(
    X_val_v2
)[:, 1]

xgb_results_5, xgb_best_5 = evaluate_thresholds(
    y_val_v2,
    xgb_prob_5,
    "V2 XGBoost 5"
)

print(xgb_best_5)

model        V2 XGBoost 5
threshold            0.63
precision        0.972099
recall           0.868791
f05              0.949518
Name: 13, dtype: object


In [57]:
# ============================================================
# EXPERIMENT 4: LENGTH DIFFERENCE FEATURES
# ============================================================

v2_feature_table_v3 = v2_feature_table.with_columns([
    (
        pl.col("name_length_ratio")
    ).alias("name_length_ratio_v3"),

    (
        pl.col("address_length_ratio")
    ).alias("address_length_ratio_v3"),
])

In [58]:
# ============================================================
# EXPERIMENT 6: XGB-3 + XGB-5 PROBABILITY BLEND
# ============================================================

blend_prob = (
    0.5 * xgb_prob_3
    + 0.5 * xgb_prob_5
)

blend_results, blend_best = evaluate_thresholds(
    y_val_v2,
    blend_prob,
    "XGB-3 + XGB-5 Blend"
)

print(blend_best)

model        XGB-3 + XGB-5 Blend
threshold                   0.66
precision               0.972822
recall                  0.864239
f05                     0.948976
Name: 16, dtype: object


In [59]:
# ============================================================
# EXPERIMENT 7: XGB-5 HEAVY BLEND
# ============================================================

blend_prob_2 = (
    0.25 * xgb_prob_3
    + 0.75 * xgb_prob_5
)

blend_results_2, blend_best_2 = evaluate_thresholds(
    y_val_v2,
    blend_prob_2,
    "XGB-3 25% + XGB-5 75%"
)

print(blend_best_2)

model        XGB-3 25% + XGB-5 75%
threshold                     0.64
precision                 0.972469
recall                    0.866812
f05                       0.949326
Name: 14, dtype: object


In [60]:
# ============================================================
# EXPERIMENT 8: FINE THRESHOLD SEARCH FOR XGB-5
# ============================================================

fine_xgb5 = []

for threshold in np.arange(0.55, 0.751, 0.0025):

    predictions = (
        xgb_prob_5 >= threshold
    ).astype(int)

    precision = precision_score(
        y_val_v2,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_val_v2,
        predictions,
        zero_division=0
    )

    f05 = fbeta_score(
        y_val_v2,
        predictions,
        beta=0.5,
        zero_division=0
    )

    fine_xgb5.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f05": f05
    })

fine_xgb5 = pd.DataFrame(fine_xgb5)

best_fine_xgb5 = fine_xgb5.loc[
    fine_xgb5["f05"].idxmax()
]

print(best_fine_xgb5)

threshold    0.645000
precision    0.972691
recall       0.867010
f05          0.949543
Name: 38, dtype: float64


In [61]:
# ============================================================
# FINAL VALIDATION COMPARISON
# ============================================================

final_candidates = [
    ("V1", v1_val_prob, 0.95),
    ("XGB-3", xgb_prob_3, 0.75),
    ("XGB-5", xgb_prob_5, 0.645),
]

comparison = []

for name, probs, threshold in final_candidates:

    pred = (probs >= threshold).astype(int)

    precision = precision_score(
        y_val_v2,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_val_v2,
        pred,
        zero_division=0
    )

    f05 = fbeta_score(
        y_val_v2,
        pred,
        beta=0.5,
        zero_division=0
    )

    tp = np.sum(
        (pred == 1) & (np.asarray(y_val_v2) == 1)
    )

    fp = np.sum(
        (pred == 1) & (np.asarray(y_val_v2) == 0)
    )

    fn = np.sum(
        (pred == 0) & (np.asarray(y_val_v2) == 1)
    )

    comparison.append({
        "model": name,
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f05": f05,
        "predicted_matches": int(pred.sum()),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn)
    })

comparison_df = pd.DataFrame(comparison)

print(comparison_df.to_string(index=False))

model  threshold  precision   recall      f05  predicted_matches   TP  FP   FN
   V1      0.950   0.959824 0.605185 0.859134               3186 3058 128 1995
XGB-3      0.750   0.976174 0.851375 0.948371               4407 4302 105  751
XGB-5      0.645   0.972691 0.867010 0.949543               4504 4381 123  672


In [62]:
# ============================================================
# FINAL MODEL: XGBOOST-5
# ============================================================

FINAL_THRESHOLD = 0.645

final_xgb = xgb.XGBClassifier(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0.05,
    reg_lambda=2.0,
    reg_alpha=0.05,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

print("Training FINAL XGBoost...")
print("Rows:", len(X_v2))
print("Features:", len(V2_FEATURES))

final_xgb.fit(
    X_v2,
    y_v2
)

print("\nFINAL MODEL TRAINED")
print("Threshold:", FINAL_THRESHOLD)

Training FINAL XGBoost...
Rows: 172458
Features: 14

FINAL MODEL TRAINED
Threshold: 0.645


In [63]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================

FINAL_MODEL_PATH = Path(
    "/kaggle/working/final_xgb_v2.joblib"
)

joblib.dump(
    final_xgb,
    FINAL_MODEL_PATH
)

print("Saved:", FINAL_MODEL_PATH)
print(
    f"Size: {FINAL_MODEL_PATH.stat().st_size / 1024**2:.2f} MB"
)

Saved: /kaggle/working/final_xgb_v2.joblib
Size: 4.41 MB


In [64]:
FINAL_CONFIG = {
    "model": "XGBoost",
    "features": V2_FEATURES,
    "threshold": FINAL_THRESHOLD,
    "n_estimators": 800,
    "max_depth": 8,
    "learning_rate": 0.03,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "min_child_weight": 3,
    "gamma": 0.05,
    "reg_lambda": 2.0,
    "reg_alpha": 0.05,
    "random_state": 42,
}

print(FINAL_CONFIG)

{'model': 'XGBoost', 'features': ['name_edit_similarity', 'name_token_jaccard', 'address_edit_similarity', 'address_token_jaccard', 'country_match', 'name_exact_normalized', 'name_exact_legal', 'name_length_ratio', 'address_exact_normalized', 'address_length_ratio', 's1_name_missing', 'match_name_missing', 's1_address_missing', 'match_address_missing'], 'threshold': 0.645, 'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.03, 'subsample': 0.85, 'colsample_bytree': 0.85, 'min_child_weight': 3, 'gamma': 0.05, 'reg_lambda': 2.0, 'reg_alpha': 0.05, 'random_state': 42}


# NEURAL NETWORK

In [65]:
# ============================================================
# EXPERIMENT NN-1: MLP NEURAL NETWORK
# ============================================================

from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

mlp_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "mlp",
        MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            batch_size=512,
            learning_rate_init=1e-3,
            max_iter=100,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,
            random_state=42
        )
    )
])

print("Training MLP...")

mlp_model.fit(
    X_train_v2,
    y_train_v2
)

print("MLP trained.")

mlp_prob = mlp_model.predict_proba(
    X_val_v2
)[:, 1]

print(
    "Iterations:",
    mlp_model.named_steps["mlp"].n_iter_
)

Training MLP...
MLP trained.
Iterations: 30


In [66]:
# ============================================================
# NN-1 THRESHOLD SEARCH
# ============================================================

mlp_results, mlp_best = evaluate_thresholds(
    y_val_v2,
    mlp_prob,
    "MLP 64-32"
)

print(mlp_best)

model        MLP 64-32
threshold          0.7
precision     0.976416
recall        0.835741
f05           0.944616
Name: 20, dtype: object


In [67]:
# ============================================================
# EXPERIMENT EXTRA TREES
# ============================================================

from sklearn.ensemble import ExtraTreesClassifier

extra_model = ExtraTreesClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

extra_model.fit(
    X_train_v2,
    y_train_v2
)

extra_prob = extra_model.predict_proba(
    X_val_v2
)[:, 1]

extra_results, extra_best = evaluate_thresholds(
    y_val_v2,
    extra_prob,
    "V2 Extra Trees"
)

print(extra_best)

model        V2 Extra Trees
threshold              0.73
precision          0.973505
recall             0.850782
f05                0.946208
Name: 23, dtype: object


In [68]:
# ============================================================
# EXPERIMENT HISTOGRAM GRADIENT BOOSTING
# ============================================================

from sklearn.ensemble import HistGradientBoostingClassifier

hist_model = HistGradientBoostingClassifier(
    max_iter=500,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=42
)

hist_model.fit(
    X_train_v2,
    y_train_v2
)

hist_prob = hist_model.predict_proba(
    X_val_v2
)[:, 1]

hist_results, hist_best = evaluate_thresholds(
    y_val_v2,
    hist_prob,
    "HistGradientBoosting"
)

print(hist_best)

model        HistGradientBoosting
threshold                    0.67
precision                0.972384
recall                   0.857115
f05                      0.946915
Name: 17, dtype: object


In [71]:
import importlib.util

print(
    "CatBoost:",
    importlib.util.find_spec("catboost") is not None
)

print(
    "LightGBM:",
    importlib.util.find_spec("lightgbm") is not None
)

CatBoost: True
LightGBM: True


In [72]:
# ============================================================
# EXPERIMENT: CATBOOST
# ============================================================

from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=800,
    depth=8,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="Logloss",
    l2_leaf_reg=3.0,
    random_seed=42,
    verbose=False,
    thread_count=-1
)

print("Training CatBoost...")

cat_model.fit(
    X_train_v2,
    y_train_v2
)

print("CatBoost trained.")

cat_prob = cat_model.predict_proba(
    X_val_v2
)[:, 1]

cat_results, cat_best = evaluate_thresholds(
    y_val_v2,
    cat_prob,
    "V2 CatBoost"
)

print(cat_best)

Training CatBoost...
CatBoost trained.
model        V2 CatBoost
threshold           0.73
precision       0.977174
recall          0.847219
f05             0.948089
Name: 23, dtype: object


In [73]:
# ============================================================
# EXPERIMENT: LIGHTGBM
# ============================================================

from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=8,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2.0,
    reg_alpha=0.05,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

print("Training LightGBM...")

lgb_model.fit(
    X_train_v2,
    y_train_v2
)

print("LightGBM trained.")

lgb_prob = lgb_model.predict_proba(
    X_val_v2
)[:, 1]

lgb_results, lgb_best = evaluate_thresholds(
    y_val_v2,
    lgb_prob,
    "V2 LightGBM"
)

print(lgb_best)

Training LightGBM...
LightGBM trained.
model        V2 LightGBM
threshold            0.7
precision       0.974695
recall           0.85375
f05              0.94784
Name: 20, dtype: object


In [75]:
print(type(training_enriched))
print(len(training_enriched))

if len(training_enriched) > 0:
    print(type(training_enriched[0]))
    print(training_enriched[0])

<class 'polars.dataframe.frame.DataFrame'>
172458
<class 'polars.dataframe.frame.DataFrame'>
shape: (1, 9)
┌────────────┬────────────┬───────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ source1_en ┆ match_id   ┆ label ┆ s1_name    ┆ … ┆ s1_countr ┆ match_nam ┆ match_add ┆ match_cou │
│ tity_id    ┆ ---        ┆ ---   ┆ ---        ┆   ┆ y         ┆ e         ┆ ress      ┆ ntry      │
│ ---        ┆ str        ┆ i32   ┆ str        ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str        ┆            ┆       ┆            ┆   ┆ str       ┆ str       ┆ str       ┆ str       │
╞════════════╪════════════╪═══════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ S1-1239063 ┆ S2-9003859 ┆ 1     ┆ Smith      ┆ … ┆ US        ┆ SMITH     ┆ 10740     ┆ US        │
│ 1          ┆ 0          ┆       ┆ Rate, LLC  ┆   ┆           ┆ RATE, LLC ┆ 200, ALEX ┆           │
│            ┆            ┆       ┆            ┆   ┆           ┆           ┆ ANDRIA, 

In [76]:
print(training_enriched.columns)

['source1_entity_id', 'match_id', 'label', 's1_name', 's1_address', 's1_country', 'match_name', 'match_address', 'match_country']


# TF-IDF

In [77]:
# ============================================================
# EXPERIMENT: TF-IDF FEATURES
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("Building TF-IDF features...")

Building TF-IDF features...


In [78]:
# ------------------------------------------------------------
# NAME TF-IDF
# ------------------------------------------------------------

train_names_s1 = (
    training_enriched["s1_name"]
    .fill_null("")
    .to_list()
)

train_names_match = (
    training_enriched["match_name"]
    .fill_null("")
    .to_list()
)

# Fit ONLY on training-side text
name_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=100000
)

name_vectorizer.fit(
    train_names_s1[:len(X_train_v2)] +
    train_names_match[:len(X_train_v2)]
)

print(
    "Name vocabulary:",
    len(name_vectorizer.vocabulary_)
)

Name vocabulary: 42253


In [79]:
# ============================================================
# GET FIXED TRAIN / VALIDATION TEXT
# ============================================================

train_idx_np = np.asarray(train_idx)
val_idx_np = np.asarray(val_idx)

s1_names = (
    training_enriched["s1_name"]
    .fill_null("")
    .to_numpy()
)

match_names = (
    training_enriched["match_name"]
    .fill_null("")
    .to_numpy()
)

s1_addresses = (
    training_enriched["s1_address"]
    .fill_null("")
    .to_numpy()
)

match_addresses = (
    training_enriched["match_address"]
    .fill_null("")
    .to_numpy()
)

train_s1_names = s1_names[train_idx_np]
train_match_names = match_names[train_idx_np]

val_s1_names = s1_names[val_idx_np]
val_match_names = match_names[val_idx_np]

train_s1_addresses = s1_addresses[train_idx_np]
train_match_addresses = match_addresses[train_idx_np]

print("Training text rows:", len(train_s1_names))
print("Validation text rows:", len(val_s1_names))

Training text rows: 137966
Validation text rows: 34492


In [80]:
# ============================================================
# TF-IDF EXPERIMENT - FIT VECTORIZERS
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

print("Fitting TF-IDF vectorizers...")

# -------------------------
# NAME TF-IDF
# -------------------------

name_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=100_000
)

name_vectorizer.fit(
    list(train_s1_names) +
    list(train_match_names)
)

print(
    "Name TF-IDF vocabulary:",
    len(name_vectorizer.vocabulary_)
)


# -------------------------
# ADDRESS TF-IDF
# -------------------------

address_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=100_000
)

address_vectorizer.fit(
    list(train_s1_addresses) +
    list(train_match_addresses)
)

print(
    "Address TF-IDF vocabulary:",
    len(address_vectorizer.vocabulary_)
)

print("\nTF-IDF vectorizers fitted.")

Fitting TF-IDF vectorizers...
Name TF-IDF vocabulary: 42731
Address TF-IDF vocabulary: 100000

TF-IDF vectorizers fitted.


In [82]:
# ============================================================
# CREATE VALIDATION ADDRESS ARRAYS
# ============================================================

val_s1_addresses = s1_addresses[val_idx_np]
val_match_addresses = match_addresses[val_idx_np]

print("Validation S1 addresses:", len(val_s1_addresses))
print("Validation match addresses:", len(val_match_addresses))

Validation S1 addresses: 34492
Validation match addresses: 34492


In [83]:
# ============================================================
# TF-IDF TRANSFORMATION
# ============================================================

print("Transforming training and validation text...")

# -------------------------
# NAME
# -------------------------

train_name_s1_tfidf = name_vectorizer.transform(train_s1_names)
train_name_match_tfidf = name_vectorizer.transform(train_match_names)

val_name_s1_tfidf = name_vectorizer.transform(val_s1_names)
val_name_match_tfidf = name_vectorizer.transform(val_match_names)


# -------------------------
# ADDRESS
# -------------------------

train_address_s1_tfidf = address_vectorizer.transform(train_s1_addresses)
train_address_match_tfidf = address_vectorizer.transform(train_match_addresses)

val_address_s1_tfidf = address_vectorizer.transform(val_s1_addresses)
val_address_match_tfidf = address_vectorizer.transform(val_match_addresses)


print("Name train shape:", train_name_s1_tfidf.shape)
print("Name val shape:", val_name_s1_tfidf.shape)

print("Address train shape:", train_address_s1_tfidf.shape)
print("Address val shape:", val_address_s1_tfidf.shape)

Transforming training and validation text...
Name train shape: (137966, 42731)
Name val shape: (34492, 42731)
Address train shape: (137966, 100000)
Address val shape: (34492, 100000)


In [84]:
# ============================================================
# TF-IDF COSINE SIMILARITY
# ============================================================

def rowwise_cosine_similarity(A, B):
    """
    Cosine similarity between corresponding rows of two
    already L2-normalized sparse matrices.
    """
    return np.asarray(A.multiply(B).sum(axis=1)).ravel()


# -------------------------
# NAME
# -------------------------

train_name_tfidf_cosine = rowwise_cosine_similarity(
    train_name_s1_tfidf,
    train_name_match_tfidf
)

val_name_tfidf_cosine = rowwise_cosine_similarity(
    val_name_s1_tfidf,
    val_name_match_tfidf
)


# -------------------------
# ADDRESS
# -------------------------

train_address_tfidf_cosine = rowwise_cosine_similarity(
    train_address_s1_tfidf,
    train_address_match_tfidf
)

val_address_tfidf_cosine = rowwise_cosine_similarity(
    val_address_s1_tfidf,
    val_address_match_tfidf
)


print("TF-IDF similarities calculated.")

print("\nName TF-IDF:")
print("Train min:", train_name_tfidf_cosine.min())
print("Train max:", train_name_tfidf_cosine.max())
print("Train mean:", train_name_tfidf_cosine.mean())

print("\nAddress TF-IDF:")
print("Train min:", train_address_tfidf_cosine.min())
print("Train max:", train_address_tfidf_cosine.max())
print("Train mean:", train_address_tfidf_cosine.mean())

TF-IDF similarities calculated.

Name TF-IDF:
Train min: 0.0
Train max: 1.0000000000000007
Train mean: 0.12079601741195445

Address TF-IDF:
Train min: 0.0
Train max: 1.0000000000000007
Train mean: 0.16809728286010636


In [85]:
# ============================================================
# TF-IDF FEATURE CHECK
# ============================================================

print("NAME TF-IDF COSINE")
print(
    "Positive mean:",
    train_name_tfidf_cosine[np.asarray(y_train_v2) == 1].mean()
)
print(
    "Negative mean:",
    train_name_tfidf_cosine[np.asarray(y_train_v2) == 0].mean()
)

print("\nADDRESS TF-IDF COSINE")
print(
    "Positive mean:",
    train_address_tfidf_cosine[np.asarray(y_train_v2) == 1].mean()
)
print(
    "Negative mean:",
    train_address_tfidf_cosine[np.asarray(y_train_v2) == 0].mean()
)

NAME TF-IDF COSINE
Positive mean: 0.6521438283329731
Negative mean: 0.02958173103076795

ADDRESS TF-IDF COSINE
Positive mean: 0.6548928112287438
Negative mean: 0.08453110300376727


In [86]:
# ============================================================
# TF-IDF ROW-WISE COSINE SIMILARITY
# ============================================================

def rowwise_cosine_similarity(A, B):
    """
    Cosine similarity between corresponding rows.
    TF-IDF vectors are already L2-normalized by default,
    so row-wise dot product = cosine similarity.
    """
    return np.asarray(
        A.multiply(B).sum(axis=1)
    ).ravel()


# ------------------------------------------------------------
# NAME
# ------------------------------------------------------------

train_name_tfidf_cosine = rowwise_cosine_similarity(
    train_name_s1_tfidf,
    train_name_match_tfidf
)

val_name_tfidf_cosine = rowwise_cosine_similarity(
    val_name_s1_tfidf,
    val_name_match_tfidf
)


# ------------------------------------------------------------
# ADDRESS
# ------------------------------------------------------------

train_address_tfidf_cosine = rowwise_cosine_similarity(
    train_address_s1_tfidf,
    train_address_match_tfidf
)

val_address_tfidf_cosine = rowwise_cosine_similarity(
    val_address_s1_tfidf,
    val_address_match_tfidf
)


print("TF-IDF similarities calculated.")

print("\nShapes:")
print("Train name   :", train_name_tfidf_cosine.shape)
print("Val name     :", val_name_tfidf_cosine.shape)
print("Train address:", train_address_tfidf_cosine.shape)
print("Val address  :", val_address_tfidf_cosine.shape)

TF-IDF similarities calculated.

Shapes:
Train name   : (137966,)
Val name     : (34492,)
Train address: (137966,)
Val address  : (34492,)


In [87]:
# ============================================================
# TF-IDF SIGNAL CHECK
# ============================================================

y_train_np = np.asarray(y_train_v2)

print("NAME TF-IDF COSINE")
print(
    "Positive mean:",
    train_name_tfidf_cosine[y_train_np == 1].mean()
)
print(
    "Negative mean:",
    train_name_tfidf_cosine[y_train_np == 0].mean()
)

print("\nADDRESS TF-IDF COSINE")
print(
    "Positive mean:",
    train_address_tfidf_cosine[y_train_np == 1].mean()
)
print(
    "Negative mean:",
    train_address_tfidf_cosine[y_train_np == 0].mean()
)

NAME TF-IDF COSINE
Positive mean: 0.6521438283329731
Negative mean: 0.02958173103076795

ADDRESS TF-IDF COSINE
Positive mean: 0.6548928112287438
Negative mean: 0.08453110300376727


In [88]:
# ============================================================
# V3 = V2 + TF-IDF
# ============================================================

X_train_v3 = np.column_stack([
    X_train_v2,
    train_name_tfidf_cosine,
    train_address_tfidf_cosine
])

X_val_v3 = np.column_stack([
    X_val_v2,
    val_name_tfidf_cosine,
    val_address_tfidf_cosine
])

V3_FEATURES = V2_FEATURES + [
    "name_tfidf_cosine",
    "address_tfidf_cosine"
]

print("V3 feature count:", len(V3_FEATURES))
print("X_train_v3:", X_train_v3.shape)
print("X_val_v3:", X_val_v3.shape)

V3 feature count: 16
X_train_v3: (137966, 16)
X_val_v3: (34492, 16)


In [89]:
# ============================================================
# EXPERIMENT: V3 TF-IDF + XGBOOST
# ============================================================

from xgboost import XGBClassifier

xgb_v3 = XGBClassifier(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0,
    reg_alpha=0.05,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("Training V3 XGBoost...")

xgb_v3.fit(
    X_train_v3,
    y_train_v2
)

print("V3 XGBoost trained.")

Training V3 XGBoost...
V3 XGBoost trained.


In [90]:
# ============================================================
# V3 VALIDATION PREDICTIONS
# ============================================================

xgb_v3_prob = xgb_v3.predict_proba(
    X_val_v3
)[:, 1]

print("V3 validation predictions generated.")

print("Count :", len(xgb_v3_prob))
print("Min   :", xgb_v3_prob.min())
print("Max   :", xgb_v3_prob.max())
print("Mean  :", xgb_v3_prob.mean())

V3 validation predictions generated.
Count : 34492
Min   : 6.667221e-07
Max   : 0.99998724
Mean  : 0.14670417


In [91]:
# ============================================================
# V3 THRESHOLD SEARCH
# ============================================================

xgb_v3_results, xgb_v3_best = evaluate_thresholds(
    y_val_v2,
    xgb_v3_prob,
    "V3 TF-IDF + XGB"
)

print(xgb_v3_best)

model        V3 TF-IDF + XGB
threshold               0.86
precision           0.990666
recall              0.966159
f05                 0.985665
Name: 36, dtype: object


In [92]:
# ============================================================
# EXPERIMENT: CHARACTER TF-IDF
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

print("Fitting character TF-IDF...")

char_name_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=150_000
)

char_address_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=150_000
)

char_name_vectorizer.fit(
    list(train_s1_names) +
    list(train_match_names)
)

char_address_vectorizer.fit(
    list(train_s1_addresses) +
    list(train_match_addresses)
)

print(
    "Name char vocabulary:",
    len(char_name_vectorizer.vocabulary_)
)

print(
    "Address char vocabulary:",
    len(char_address_vectorizer.vocabulary_)
)

Fitting character TF-IDF...
Name char vocabulary: 150000
Address char vocabulary: 150000


In [93]:
# ============================================================
# CHARACTER TF-IDF TRANSFORMATION
# ============================================================

print("Transforming character TF-IDF...")

train_char_name_s1 = char_name_vectorizer.transform(train_s1_names)
train_char_name_match = char_name_vectorizer.transform(train_match_names)

val_char_name_s1 = char_name_vectorizer.transform(val_s1_names)
val_char_name_match = char_name_vectorizer.transform(val_match_names)


train_char_address_s1 = char_address_vectorizer.transform(
    train_s1_addresses
)

train_char_address_match = char_address_vectorizer.transform(
    train_match_addresses
)

val_char_address_s1 = char_address_vectorizer.transform(
    val_s1_addresses
)

val_char_address_match = char_address_vectorizer.transform(
    val_match_addresses
)

print("Character TF-IDF transformation complete.")

Transforming character TF-IDF...
Character TF-IDF transformation complete.


In [94]:
# ============================================================
# CHARACTER TF-IDF COSINE FEATURES
# ============================================================

train_name_char_tfidf_cosine = rowwise_cosine_similarity(
    train_char_name_s1,
    train_char_name_match
)

val_name_char_tfidf_cosine = rowwise_cosine_similarity(
    val_char_name_s1,
    val_char_name_match
)

train_address_char_tfidf_cosine = rowwise_cosine_similarity(
    train_char_address_s1,
    train_char_address_match
)

val_address_char_tfidf_cosine = rowwise_cosine_similarity(
    val_char_address_s1,
    val_char_address_match
)

print("Character TF-IDF similarities calculated.")

print(
    "Name mean:",
    train_name_char_tfidf_cosine.mean()
)

print(
    "Address mean:",
    train_address_char_tfidf_cosine.mean()
)

Character TF-IDF similarities calculated.
Name mean: 0.13945017329546125
Address mean: 0.22278123414448564


In [99]:
# ============================================================
# V4 = V3 + CHARACTER TF-IDF
# ============================================================

X_train_v4 = np.column_stack([
    X_train_v3,
    train_name_char_tfidf_cosine,
    train_address_char_tfidf_cosine
])

X_val_v4 = np.column_stack([
    X_val_v3,
    val_name_char_tfidf_cosine,
    val_address_char_tfidf_cosine
])

V4_FEATURES = V3_FEATURES + [
    "name_char_tfidf_cosine",
    "address_char_tfidf_cosine"
]

print("V4 feature count:", len(V4_FEATURES))
print("X_train_v4:", X_train_v4.shape)
print("X_val_v4:", X_val_v4.shape)

V4 feature count: 18
X_train_v4: (137966, 18)
X_val_v4: (34492, 18)


In [100]:
# ============================================================
# V4 CHARACTER TF-IDF + XGB
# ============================================================

xgb_v4 = XGBClassifier(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0,
    reg_alpha=0.05,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("Training V4 XGBoost...")

xgb_v4.fit(
    X_train_v4,
    y_train_v2
)

print("V4 XGBoost trained.")

Training V4 XGBoost...
V4 XGBoost trained.


In [ ]:
# ============================================================
# V4 PREDICTIONS + THRESHOLD SEARCH
# ============================================================

xgb_v4_prob = xgb_v4.predict_proba(
    X_val_v4
)[:, 1]

v4_results, v4_best = evaluate_thresholds(
    y_val_v2,
    xgb_v4_prob,
    "V4 Char-TFIDF + XGB"
)

print(v4_best)

In [101]:
# ============================================================
# V4 PREDICTIONS + THRESHOLD SEARCH
# ============================================================

xgb_v4_prob = xgb_v4.predict_proba(
    X_val_v4
)[:, 1]

v4_results, v4_best = evaluate_thresholds(
    y_val_v2,
    xgb_v4_prob,
    "V4 Char-TFIDF + XGB"
)

print(v4_best)

model        V4 Char-TFIDF + XGB
threshold                    0.9
precision               0.992527
recall                  0.972492
f05                     0.988454
Name: 40, dtype: object


### V4 Checkpoint

In [102]:
# ============================================================
# V4 CHECKPOINT
# ============================================================

v4_checkpoint = {
    "model": xgb_v4,
    "threshold": float(v4_best["threshold"]),
    "features": V4_FEATURES,
    "best_metrics": v4_best.to_dict(),
}

print("V4 checkpoint:")
print(v4_best)
print("\nFeatures:", V4_FEATURES)

V4 checkpoint:
model        V4 Char-TFIDF + XGB
threshold                    0.9
precision               0.992527
recall                  0.972492
f05                     0.988454
Name: 40, dtype: object

Features: ['name_edit_similarity', 'name_token_jaccard', 'address_edit_similarity', 'address_token_jaccard', 'country_match', 'name_exact_normalized', 'name_exact_legal', 'name_length_ratio', 'address_exact_normalized', 'address_length_ratio', 's1_name_missing', 'match_name_missing', 's1_address_missing', 'match_address_missing', 'name_tfidf_cosine', 'address_tfidf_cosine', 'name_char_tfidf_cosine', 'address_char_tfidf_cosine']


In [104]:
import joblib

joblib.dump(
    xgb_v4,
    "/kaggle/working/xgb_v4_char_tfidf.joblib"
)

print("V4 model saved.")

V4 model saved.
